# 04. Knowledge Synchronization Lifecycle

Tests the complete document lifecycle: UPLOAD -> VERIFY -> UPDATE -> PURGE OLD VECTORS -> DELETE -> VERIFY SYNC.


In [ ]:
# Setup environment and path
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from app.core.config import get_settings
settings = get_settings()
print(f'RAG Environment: {settings.RAG_ENV}, Canonical Collection: {settings.CANONICAL_COLLECTION}')


In [ ]:
import asyncio
from app.ingestion.pipeline import ingest_text, verify_document_index
from app.vectorstore.chroma import get_unified_collection, delete_document_chunks

doc_id = 'DOC_SYNC_TEST_01'
text_v1 = 'Version 1: Database normalization eliminates redundancy using 1NF, 2NF, 3NF, BCNF.'

# 1. Ingest V1
res_v1 = asyncio.run(ingest_text(text_v1, doc_id, document_version=1))
print('V1 Ingestion:', res_v1.status, f'chunks={res_v1.chunk_count}')

# 2. Verify Index Sync
sync_v1 = verify_document_index(doc_id)
print('V1 Sync Verification:', sync_v1)

# 3. Update to V2 (Old vectors purged, new indexed)
text_v2 = 'Version 2: Database normalization eliminates redundancy and guarantees lossless decomposition.'
res_v2 = asyncio.run(ingest_text(text_v2, doc_id, document_version=2))
print('V2 Ingestion:', res_v2.status, f'chunks={res_v2.chunk_count}')

# 4. Clean up / Delete
coll = get_unified_collection()
delete_document_chunks(coll, doc_id)
print('Document purged. Final Sync Check:', verify_document_index(doc_id))
